In [ ]:
from torchgeo.trainers import PixelwiseRegressionTask
import torch
import pytorch_lightning as pl
import numpy as np
import rasterio
import cv2
from torch.utils.data import Dataset, DataLoader
from typing import List
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint
import torch.nn as nn
import os
from utils.data.LandsatDataModule import LandsatDataModule, LandsatDataset

class LSTNowcaster(pl.LightningModule):
    def __init__(self, in_channels=5, learning_rate=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = PixelwiseRegressionTask(
            model="unet",
            backbone="resnet50",
            weights=True,
            in_channels=in_channels,
            num_outputs=1,
            loss="mse",
            lr=learning_rate
        )
        self.criterion = nn.MSELoss()
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']

        outputs = self(inputs)
        loss = self.criterion(outputs[mask], targets[mask])

        self.log('train_loss', loss,
                 on_step=True,
                 on_epoch=True,
                 prog_bar=True,
                 sync_dist=True)  # Add this parameter
        return loss

    def validation_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']

        outputs = self(inputs)
        loss = self.criterion(outputs[mask], targets[mask])

        self.log('val_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']
        profile = batch['profile']
        file_path = batch['file_path']

        outputs = self(inputs)
        mse = self.criterion(outputs[mask], targets[mask])

        # Save prediction if it's the first batch
        if batch_idx == 0:
            predicted = outputs.cpu().numpy().squeeze()
            mask_np = mask.cpu().numpy().squeeze()
            predicted[~mask_np] = np.nan

            profile = profile[0]  # Get first item's profile
            profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

            output_filename = f"predicted_LST_batch_{batch_idx}.tif"
            with rasterio.open(output_filename, "w", **profile) as dst:
                dst.write(predicted.astype(np.float32), 1)

            print(f"Saved predicted LST raster: {output_filename}")

        return {'test_loss': mse, 'batch_size': inputs.size(0)}

    def test_epoch_end(self, outputs):
        avg_mse = torch.stack([x['test_loss'] for x in outputs]).mean()
        rmse = torch.sqrt(avg_mse)

        self.log('test_mse', avg_mse)
        self.log('test_rmse', rmse)

        print(f"Test RMSE: {rmse:.4f}")

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

def main():
    import logging
    def device_info_filter(record):
        return "PU available: " not in record.getMessage()
    logging.getLogger("lightning.pytorch.utilities.rank_zero").addFilter(device_info_filter)

    # Initialize data module
    data_module = LandsatDataModule(
        data_dir="./Data",
        batch_size=1,
        num_workers=0,
        debug=True
    )

    # Initialize trainer with explicit steps
    trainer = pl.Trainer(
        max_epochs=200,
        gradient_clip_val=0.5,
        log_every_n_steps=1,
        enable_progress_bar=True,
        enable_model_summary=True,
        deterministic=True,
        num_sanity_val_steps=2,
        reload_dataloaders_every_n_epochs=1
    )

    model = LSTNowcaster(in_channels=5, learning_rate=1e-4)

    # Train model
    trainer.fit(model=model, datamodule=data_module)

    # Test model
    trainer.test(model=model, datamodule=data_module)

import warnings
if __name__ == "__main__":
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="TORCH_MODEL_ZOO is deprecated")
    main()

KeyboardInterrupt: 

In [ ]:
def save_prediction_and_truth(model, test_loader, test_file_list, device):
    """
    Save both prediction and ground truth from test loader as georeferenced TIFFs
    """
    model.eval()

    with torch.no_grad():
        # Get one sample
        sample = next(iter(test_loader))
        inputs = sample['input'].to(device)
        targets = sample['target'].to(device)
        mask = sample['mask'].to(device)

        # Get corresponding LST file path
        lst_tif_path = test_file_list[0]['LST.tif']

        # Convert tensors to numpy arrays
        mask_np = mask.cpu().numpy().squeeze()
        targets_np = targets.cpu().numpy().squeeze()

        # Get model prediction
        outputs = model(inputs)
        predicted_np = outputs.cpu().numpy().squeeze()

        # Apply mask to both prediction and ground truth
        predicted_np[~mask_np] = np.nan
        targets_np[~mask_np] = np.nan

        # Get geospatial metadata from original LST file
        with rasterio.open(lst_tif_path) as src:
            profile = src.profile.copy()
            profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

            # Denormalize values back to Fahrenheit
            # Using the same range as in the Normalize class
            predicted_np = predicted_np * (250 - (-50)) + (-50)  # Updated range
            targets_np = targets_np * (250 - (-50)) + (-50)      # Updated range

            # Save prediction
            pred_filename = "predicted_LST.tif"
            with rasterio.open(pred_filename, "w", **profile) as dst:
                dst.write(predicted_np.astype(np.float32), 1)

            # Save ground truth
            truth_filename = "ground_truth_LST.tif"
            with rasterio.open(truth_filename, "w", **profile) as dst:
                dst.write(targets_np.astype(np.float32), 1)

            # Calculate and print some statistics for valid pixels
            valid_mask = ~np.isnan(predicted_np)
            if valid_mask.any():
                mae = np.mean(np.abs(predicted_np[valid_mask] - targets_np[valid_mask]))
                rmse = np.sqrt(np.mean((predicted_np[valid_mask] - targets_np[valid_mask])**2))
                print(f"Mean Absolute Error: {mae:.2f}°F")
                print(f"Root Mean Square Error: {rmse:.2f}°F")

        print(f"Saved files:")
        print(f"Predictions: {pred_filename}")
        print(f"Ground Truth: {truth_filename}")
        print(f"Original LST: {lst_tif_path}")

In [ ]:
    # Assume model and test_loader are already defined
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load the trained model
    checkpoint = torch.load('best_model.pth')
    model = UNet(n_channels=5, bilinear=False).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    save_prediction_and_truth(model, test_loader, test_file_list, device)